스포츠센터 고객 데이터 분석 시나리오
```
제가 운영하는 스포츠센터는 트레이닝 붐 덕분에 지금까지 고객 수가 늘었습니다.
그런데 최근 1년간 고객 수가 늘지 않는 것 같습니다.
자주 이용하는 고객은 계속 오지만, 가끔 오는 고객은 어느새 오지 않는 경우도 생기는 것 같습니다.
제대로 데이터를 분석한 적이 없어서 어떤 고객이 계속 이용하고 있는지조차 모릅니다.
데이터분석을 하면 뭔가 알 수 있을까요?

스포츠센터에는 종일권, 주간권, 야간권으로 3종류의 회원 구분이 있습니다.
또, 일반적으로 입회비가 있지만, 비정기적으로 입회비 반액 할인이나 입회비 무료 행사를 진행해서 신규회원을 늘리고 있습니다.
탈퇴하려면 월말까지 신청하면 그 다음 달 말에 탈퇴가 됩니다.

2018년 4월 ~ 2019년 3월까지 1년 데이터
```

```
customer_master2.csv : 2019년 3월 말 시점의 회원 데이터 (이전 탈퇴 회원 포함)
use_log.csv : 이용이력, 기간은 2018.04~2019.03
class_master.csv : 회원 구분 데이터(종일, 주간, 야간)
campaign_master.csv : 행사 구분 데이터(입회비 유무 등)
```
| 파일                 | 한마디로              |
| ------------------ | ----------------- |
| `customer_master2` | **회원은 누구인가?**     |
| `use_log`          | **회원이 언제 방문했나?**  |
| `class_master`     | **어떤 회원권을 쓰나?**   |
| `campaign_master`  | **어떤 할인으로 가입했나?** |
```
customer_master2.csv  → 회원 정보
use_log.csv           → 회원의 이용 기록
class_master.csv      → 회원권 종류
campaign_master.csv   → 가입 행사 종류
```
| 파일명                    | 내용                                                  |
| ---------------------- | --------------------------------------------------- |
| `customer_master2.csv` | **2019년 3월 말 기준 회원 데이터**이며, 현재 회원뿐 아니라 이전 탈퇴 회원도 포함 |
| `use_log.csv`          | **회원 이용 이력 데이터**이며, 기간은 **2018년 4월 ~ 2019년 3월**     |
| `class_master.csv`     | **회원 구분 데이터**로, 종일·주간·야간 회원권 정보를 관리                 |
| `campaign_master.csv`  | **가입 행사 구분 데이터**로, 입회비 무료·반액 할인 등의 행사 정보를 관리        |

데이터의 최종 목적은 회원의 이용 패턴을 분석해서 “어떤 회원이 탈퇴할 가능성이 높은가?” 같은 것을 분석하는 것

customer_master2.csv — 1 회원 기본정보
##### 총 4,192명 × 8개 컬럼으로, 핵심이 되는 회원 데이터
| 컬럼            | 의미        |
| ------------- | --------- |
| `customer_id` | 회원 고유번호   |
| `name`        | 회원 이름     |
| `class`       | 회원권 종류 코드 |
| `gender`      | 성별        |
| `start_date`  | 가입일       |
| `end_date`    | 탈퇴일       |
| `campaign_id` | 가입 당시 캠페인 |
| `is_deleted`  | 탈퇴 여부     |
```
is_deleted = 0 → 현재 회원
is_deleted = 1 → 탈퇴 회원
end_date가 비어 있음 → 아직 탈퇴하지 않은 회원

→ 누가 언제 가입했고, 어떤 회원권을 사용하며, 현재 유지 중인지 탈퇴했는지를 가지고 있는 핵심 파일

use_log.csv — 2 회원 이용기록
##### 총 197,428건 × 3개 컬럼
| 컬럼            | 의미        |
| ------------- | --------- |
| `log_id`      | 이용기록 번호   |
| `customer_id` | 이용한 회원 번호 |
| `usedate`     | 이용 날짜     |


class_master.csv — 3 회원권 종류 및 가격
| 컬럼           | 의미     | 예시           |
| ------------ | ------ | ------------ |
| `class`      | 회원권 코드 | C01          |
| `class_name` | 회원권 종류 | 종일 / 주간 / 야간 |
| `price`      | 월 이용가격 | 10,500       |
회원이 어떤 이용권을 사용하는지와 가격을 관리하는 파일

campaign_master.csv — 4 가입 캠페인 정보
| 컬럼              | 의미           | 예시                   |
| --------------- | ------------ | -------------------- |
| `campaign_id`   | 캠페인 코드       | CA1                  |
| `campaign_name` | 가입 할인/이벤트 종류 | 일반, 입회비 반액할인, 입회비 무료 |


## 전체 분석 흐름
```text
1. 고객정보 불러오기
        ↓
2. 회원권/가입행사 정보 붙이기
        ↓
3. 고객 현황 확인
        ↓
4. 이용기록에서 월별 방문횟수 계산
        ↓
5. 규칙적으로 오는 회원인지 확인
        ↓
6. 고객정보 + 이용정보 합치기
        ↓
7. 탈퇴회원과 유지회원 비교
        ↓
"탈퇴하는 회원은 어떤 특징이 있을까?"
```


> 핵심: **고객정보를 정리하고 → 방문기록을 정리하고 → 둘을 합쳐서 → 탈퇴회원과 유지회원의 차이를 비교한다.**


In [1]:
# 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

import os

# 실습 폴더로 이동
os.chdir('/content/drive/MyDrive/Colab Notebooks/Data_Analysis/DA_01')

# 현재 위치 확인
print(os.getcwd())


Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/Data_Analysis/DA_01


In [2]:
%run start.py

## 이번 노트북에서 할 일
- 고객정보를 정리한다.
- 월별 이용횟수를 계산한다.
- 규칙적으로 이용하는 회원인지 표시한다.
- 고객정보와 이용정보를 합친다.
- 탈퇴회원과 유지회원의 차이를 비교한다.


###데이터읽기

In [3]:
# 회원이 언제 센터를 이용했는지 기록한 파일
uselog = pd.read_csv('data/use_log.csv')

# 전체 이용기록 수
print(len(uselog))

# 앞부분 2개 확인
uselog.head(2)


197428


,log_id,customer_id,usedate
0,L00000049012330,AS009373,2018-04-01
1,L00000049012331,AS015315,2018-04-01


In [4]:
# 회원권 정보(종일권/주간권/야간권)
class_master = pd.read_csv('data/class_master.csv')

print(len(class_master))
class_master


3


,class,class_name,price
0,C01,0_종일,10500
1,C02,1_주간,7500
2,C03,2_야간,6000


In [5]:
# 고객(회원) 기본정보
customer = pd.read_csv('data/customer_master2.csv')

print(len(customer))
customer.head(2)


4192


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted
0,OA832399,XXXX,C01,F,2015-05-01 00:00:00,NaN,CA1,0
1,PL270116,XXXXX,C01,M,2015-05-01 00:00:00,NaN,CA1,0


In [6]:
# 가입 행사정보(일반/입회비 반액/입회비 무료)
campaign_master = pd.read_csv('data/campaign_master.csv')

print(len(campaign_master))
campaign_master


3


,campaign_id,campaign_name
0,CA1,2_일반
1,CA2,0_입회비반액할인
2,CA3,1_입회비무료


#### 고객 데이터 가공

## 데이터 결합
- 실제 업무 데이터는 한 파일에 모든 정보가 들어 있지 않은 경우가 많다.
- 그래서 고객정보 + 이용이력 + 상품정보처럼 여러 표를 연결해야 한다.
- `merge()`는 공통 열(Key)을 기준으로 두 DataFrame을 연결하는 함수이다.
- 데이터베이스의 **JOIN**과 거의 같은 개념으로 생각하면 된다


In [7]:
# 고객정보 + 회원권정보 합치기
# 공통 컬럼인 class를 기준으로 연결
# how="left" → 왼쪽 customer를 기준으로 모두 남김
customer_join1 = pd.merge(
    customer,
    class_master,
    on="class",
    how="left"
)

customer_join1.head()

# customer + class_master → customer_join1


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price
0,OA832399,XXXX,C01,F,2015-05-01 00:00:00,NaN,CA1,0,0_종일,10500
1,PL270116,XXXXX,C01,M,2015-05-01 00:00:00,NaN,CA1,0,0_종일,10500
2,OA974876,XXXXX,C01,M,2015-05-01 00:00:00,NaN,CA1,0,0_종일,10500
3,HD024127,XXXXX,C01,F,2015-05-01 00:00:00,NaN,CA1,0,0_종일,10500
4,HD661448,XXXXX,C03,F,2015-05-01 00:00:00,NaN,CA1,0,2_야간,6000


In [8]:
# 위에서 합친 고객정보에 가입 행사정보까지 붙이기
# 공통 컬럼인 campaign_id를 기준으로 연결
customer_join2 = pd.merge(
    customer_join1,
    campaign_master,
    on="campaign_id",
    how="left"
)

customer_join2.head()

# customer_join1 + campaign_master → customer_join2


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,campaign_name
0,OA832399,XXXX,C01,F,2015-05-01 00:00:00,NaN,CA1,0,0_종일,10500,2_일반
1,PL270116,XXXXX,C01,M,2015-05-01 00:00:00,NaN,CA1,0,0_종일,10500,2_일반
2,OA974876,XXXXX,C01,M,2015-05-01 00:00:00,NaN,CA1,0,0_종일,10500,2_일반
3,HD024127,XXXXX,C01,F,2015-05-01 00:00:00,NaN,CA1,0,0_종일,10500,2_일반
4,HD661448,XXXXX,C03,F,2015-05-01 00:00:00,NaN,CA1,0,2_야간,6000,2_일반


## 결측치(NaN)
- 결측치는 값이 비어 있는 상태이다.
- `isnull().sum()`으로 열별 결측치 개수를 확인할 수 있다.
- `dropna()`는 결측치가 있는 데이터를 제거하고,
- `fillna()`는 평균·중앙값 등 다른 값으로 채운다.
- 어떤 방법이 무조건 정답은 아니고 **데이터의 의미를 보고 결정**해야 한다.


In [9]:
# 컬럼별로 비어 있는 값(NaN)이 몇 개인지 확인
customer_join2.isnull().sum()


,0
customer_id,0
name,0
class,0
gender,0
start_date,0
end_date,2842
campaign_id,0
is_deleted,0
class_name,0
price,0


### 고객 데이터 집계

## groupby / agg
- `groupby()`는 같은 기준끼리 데이터를 묶는 기능이다.
- 예: 반별, 성별, 고객별, 월별로 묶기
- `agg()`는 묶은 데이터에 평균·합계·개수·최댓값 등을 한 번에 계산할 때 사용한다.
- 실무 데이터 분석에서는 **'묶고 → 요약하기'**가 매우 자주 사용된다.


In [10]:
# 회원권 종류별 고객 수 확인
customer_join2.groupby("class_name")[["customer_id"]].count()


,customer_id
class_name,
0_종일,2045
1_주간,1019
2_야간,1128


### 회원권별 탈퇴한 인원수와 금액 합

In [11]:
# 회원권 종류별 탈퇴 인원과 가격 합계 확인
# is_deleted는 탈퇴하면 1이므로 합계 = 탈퇴 인원
customer_join2.groupby("class_name")[["is_deleted", "price"]].sum()


,is_deleted,price
class_name,,
0_종일,664,21472500
1_주간,344,7642500
2_야간,342,6768000


### 캠페인별 카운트

In [12]:
# 가입 행사별 고객 수 확인
customer_join2.groupby("campaign_name")[["customer_id"]].count()


,customer_id
campaign_name,
0_입회비반액할인,650
1_입회비무료,492
2_일반,3050


### 성별 카운트

In [13]:
# 성별 고객 수 확인
customer_join2.groupby("gender")[["customer_id"]].count()


,customer_id
gender,
F,1983
M,2209


## 데이터 상태 확인
- `info()` : 열 이름, 데이터 타입, 결측치 여부 등을 확인한다.
- `describe()` : **숫자형 열 중심으로** 평균, 표준편차, 최솟값, 최댓값, 사분위수를 보여준다.
- 분석을 시작하면 모델부터 만드는 것이 아니라 **데이터 상태를 먼저 확인**하는 습관이 중요하다.


In [14]:
# 데이터의 컬럼명, 자료형, 결측치 여부 확인
customer_join2.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4192 entries, 0 to 4191
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   customer_id    4192 non-null   object
 1   name           4192 non-null   object
 2   class          4192 non-null   object
 3   gender         4192 non-null   object
 4   start_date     4192 non-null   object
 5   end_date       1350 non-null   object
 6   campaign_id    4192 non-null   object
 7   is_deleted     4192 non-null   int64 
 8   class_name     4192 non-null   object
 9   price          4192 non-null   int64 
 10  campaign_name  4192 non-null   object
dtypes: int64(2), object(9)
memory usage: 360.4+ KB


#### 2019년 3월 말 기준 회원만 확인


In [15]:
# 탈퇴일을 문자열 → 날짜형(datetime)으로 변경
# 날짜 비교를 하려면 먼저 날짜형으로 바꿔야 함
customer_join2["end_date"] = pd.to_datetime(customer_join2["end_date"])

customer_join2.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4192 entries, 0 to 4191
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    4192 non-null   object        
 1   name           4192 non-null   object        
 2   class          4192 non-null   object        
 3   gender         4192 non-null   object        
 4   start_date     4192 non-null   object        
 5   end_date       1350 non-null   datetime64[ns]
 6   campaign_id    4192 non-null   object        
 7   is_deleted     4192 non-null   int64         
 8   class_name     4192 non-null   object        
 9   price          4192 non-null   int64         
 10  campaign_name  4192 non-null   object        
dtypes: datetime64[ns](1), int64(2), object(8)
memory usage: 360.4+ KB


In [16]:
# 2019년 3월 말에 회원이었던 사람만 남기기
# 1) end_date가 없음 → 현재 회원 → 포함
# 2) end_date가 2019-03-31 이상 → 3월 말까지 회원이었음 → 포함
# 3) 그 전에 탈퇴한 회원 → 제외
customer_new = customer_join2.loc[
    customer_join2["end_date"].isna() |
    (customer_join2["end_date"] >= pd.to_datetime("2019-03-31"))
]

# 현재 회원 2,842명 + 2019-03-31 탈퇴 회원 111명 = 2,953명
print(len(customer_new))


2953


In [17]:
# 필터링한 회원 데이터 상태 확인
customer_new.info()


<class 'pandas.core.frame.DataFrame'>
Index: 2953 entries, 0 to 4191
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   customer_id    2953 non-null   object        
 1   name           2953 non-null   object        
 2   class          2953 non-null   object        
 3   gender         2953 non-null   object        
 4   start_date     2953 non-null   object        
 5   end_date       111 non-null    datetime64[ns]
 6   campaign_id    2953 non-null   object        
 7   is_deleted     2953 non-null   int64         
 8   class_name     2953 non-null   object        
 9   price          2953 non-null   int64         
 10  campaign_name  2953 non-null   object        
dtypes: datetime64[ns](1), int64(2), object(8)
memory usage: 276.8+ KB


In [18]:
# 필터링한 데이터 앞부분 확인
customer_new.head(3)


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,campaign_name
0,OA832399,XXXX,C01,F,2015-05-01 00:00:00,NaT,CA1,0,0_종일,10500,2_일반
1,PL270116,XXXXX,C01,M,2015-05-01 00:00:00,NaT,CA1,0,0_종일,10500,2_일반
2,OA974876,XXXXX,C01,M,2015-05-01 00:00:00,NaT,CA1,0,0_종일,10500,2_일반


In [19]:
# 2019년 3월 말 기준 회원을 회원권 종류별로 몇 명인지 확인
customer_new.groupby("class_name")[["customer_id"]].count()


,customer_id
class_name,
0_종일,1444
1_주간,696
2_야간,813


#### 이용 이력 데이터 (uselog)

In [20]:
# 이용이력 데이터 구조 확인
uselog.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 197428 entries, 0 to 197427
Data columns (total 3 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   log_id       197428 non-null  object
 1   customer_id  197428 non-null  object
 2   usedate      197428 non-null  object
dtypes: object(3)
memory usage: 4.5+ MB


In [21]:
# 이용일을 문자열 → 날짜형(datetime)으로 변경
uselog["usedate"] = pd.to_datetime(uselog["usedate"])

# 변환 결과 확인
uselog[["usedate"]].head()


,usedate
0,2018-04-01
1,2018-04-01
2,2018-04-01
3,2018-04-01
4,2018-04-01


In [22]:
# 이용일에서 연도+월만 뽑아 "년월" 컬럼 만들기
# 예: 2018-04-05 → 201804
uselog["년월"] = uselog["usedate"].dt.strftime("%Y%m")

uselog.head()


,log_id,customer_id,usedate,년월
0,L00000049012330,AS009373,2018-04-01,201804
1,L00000049012331,AS015315,2018-04-01,201804
2,L00000049012332,AS040841,2018-04-01,201804
3,L00000049012333,AS046594,2018-04-01,201804
4,L00000049012334,AS073285,2018-04-01,201804


In [23]:
# 월별 전체 이용 횟수 확인
uselog.groupby("년월")[["log_id"]].count()


,log_id
년월,
201804,16460
201805,16913
201806,16994
201807,16954
201808,16674
201809,16471
201810,15989
201811,15697
201812,16383


In [24]:
# 고객별 + 월별 이용 횟수 계산
# 예: A001 고객이 2018년 4월에 몇 번 왔는지 계산
uselog_months = uselog.groupby(
    ["년월", "customer_id"],
    as_index=False
)[["usedate"]].count()

uselog_months


,년월,customer_id,usedate
0,201804,AS002855,4
1,201804,AS009013,2
2,201804,AS009373,3
3,201804,AS015315,6
4,201804,AS015739,7
...,...,...,...
36837,201903,TS995853,8
36838,201903,TS998593,8
36839,201903,TS999079,3
36840,201903,TS999231,6


In [25]:
# usedate는 이제 날짜가 아니라 "이용 횟수"이므로 이름을 count로 변경
uselog_months.rename(columns={"usedate": "count"}, inplace=True)

uselog_months


,년월,customer_id,count
0,201804,AS002855,4
1,201804,AS009013,2
2,201804,AS009373,3
3,201804,AS015315,6
4,201804,AS015739,7
...,...,...,...
36837,201903,TS995853,8
36838,201903,TS998593,8
36839,201903,TS999079,3
36840,201903,TS999231,6


In [26]:
# 고객별 월 이용횟수를 요약
# mean   : 월 평균 이용횟수
# median : 중앙값
# max    : 가장 많이 이용한 달의 횟수
# min    : 가장 적게 이용한 달의 횟수
uselog_customer = uselog_months.groupby("customer_id")["count"].agg(
    ["mean", "median", "max", "min"]
)

# customer_id가 index로 들어갔으므로 다시 일반 컬럼으로 꺼내기
uselog_customer = uselog_customer.reset_index()

uselog_customer


,customer_id,mean,median,max,min
0,AS002855,4.500000,5.0,7,2
1,AS008805,4.000000,4.0,8,1
2,AS009013,2.000000,2.0,2,2
3,AS009373,5.083333,5.0,7,3
4,AS015233,7.545455,7.0,11,4
...,...,...,...,...,...
4187,TS995853,9.500000,9.5,11,8
4188,TS998593,8.142857,8.0,9,7
4189,TS999079,4.916667,5.5,9,2
4190,TS999231,4.666667,5.0,8,1


# 'customer_id'로 그룹화하고, 'count' 컬럼에 대해 'mean', 'median', 'max', 'min'을 계산
## 월별 각 회원의 이용횟수에 대한 값들

#### 규칙적으로 이용하는 회원인지 확인


In [27]:
# 이용이력 데이터 확인
uselog.head()


,log_id,customer_id,usedate,년월
0,L00000049012330,AS009373,2018-04-01,201804
1,L00000049012331,AS015315,2018-04-01,201804
2,L00000049012332,AS040841,2018-04-01,201804
3,L00000049012333,AS046594,2018-04-01,201804
4,L00000049012334,AS073285,2018-04-01,201804


In [28]:
# 이용일에서 요일 번호 만들기
# 월=0, 화=1, 수=2, 목=3, 금=4, 토=5, 일=6
uselog["weekday"] = uselog["usedate"].dt.weekday

uselog.head()


,log_id,customer_id,usedate,년월,weekday
0,L00000049012330,AS009373,2018-04-01,201804,6
1,L00000049012331,AS015315,2018-04-01,201804,6
2,L00000049012332,AS040841,2018-04-01,201804,6
3,L00000049012333,AS046594,2018-04-01,201804,6
4,L00000049012334,AS073285,2018-04-01,201804,6


In [29]:
# 고객별 + 월별 + 요일별 이용 횟수 계산
# 예: A001 고객이 4월의 월요일에 몇 번 왔는지 계산
uselog_weekday = uselog.groupby(
    ["customer_id", "년월", "weekday"],
    as_index=False
).count()[["customer_id", "년월", "weekday", "log_id"]]

uselog_weekday


,customer_id,년월,weekday,log_id
0,AS002855,201804,5,4
1,AS002855,201805,2,1
2,AS002855,201805,5,4
3,AS002855,201806,5,5
4,AS002855,201807,1,1
...,...,...,...,...
93328,TS999855,201901,1,1
93329,TS999855,201901,5,4
93330,TS999855,201901,6,1
93331,TS999855,201902,5,4


In [30]:
# log_id 개수 = 이용 횟수이므로 컬럼명을 count로 변경
uselog_weekday.rename(columns={"log_id": "count"}, inplace=True)

uselog_weekday


,customer_id,년월,weekday,count
0,AS002855,201804,5,4
1,AS002855,201805,2,1
2,AS002855,201805,5,4
3,AS002855,201806,5,5
4,AS002855,201807,1,1
...,...,...,...,...
93328,TS999855,201901,1,1
93329,TS999855,201901,5,4
93330,TS999855,201901,6,1
93331,TS999855,201902,5,4


In [31]:
# 고객마다 같은 요일에 가장 많이 방문한 횟수만 남기기
# 이 값으로 규칙적으로 오는 회원인지 판단할 예정
uselog_weekday = uselog_weekday.groupby(
    "customer_id", as_index=False
).max()[["customer_id", "count"]]

uselog_weekday


,customer_id,count
0,AS002855,5
1,AS008805,4
2,AS009013,2
3,AS009373,5
4,AS015233,5
...,...,...
4187,TS995853,5
4188,TS998593,5
4189,TS999079,5
4190,TS999231,5


In [32]:
# 규칙적으로 이용하는지 표시할 컬럼 만들기
# 기본값 0 = 불규칙 이용
uselog_weekday["routine_flg"] = 0

# 같은 요일에 한 달 4회 이상 방문한 적이 있으면
# 1 = 규칙적으로 이용하는 회원으로 표시
uselog_weekday.loc[
    uselog_weekday["count"] >= 4,
    "routine_flg"
] = 1

uselog_weekday.head()


,customer_id,count,routine_flg
0,AS002855,5,1
1,AS008805,4,1
2,AS009013,2,0
3,AS009373,5,1
4,AS015233,5,1


#### 고객정보 + 이용정보 합치기


In [33]:
# 고객별 이용횟수 통계 확인
uselog_customer.head(2)


,customer_id,mean,median,max,min
0,AS002855,4.5,5.0,7,2
1,AS008805,4.0,4.0,8,1


In [34]:
# 고객 기본정보 확인
customer_join2.head(2)


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,campaign_name
0,OA832399,XXXX,C01,F,2015-05-01 00:00:00,NaT,CA1,0,0_종일,10500,2_일반
1,PL270116,XXXXX,C01,M,2015-05-01 00:00:00,NaT,CA1,0,0_종일,10500,2_일반


In [35]:
# 고객정보 + 고객별 이용횟수 통계 합치기
# customer_id를 기준으로 연결
customer_join3 = pd.merge(
    customer_join2,
    uselog_customer,
    on="customer_id",
    how="left"
)

customer_join3.head(3)


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,campaign_name,mean,median,max,min
0,OA832399,XXXX,C01,F,2015-05-01 00:00:00,NaT,CA1,0,0_종일,10500,2_일반,4.833333,5.0,8,2
1,PL270116,XXXXX,C01,M,2015-05-01 00:00:00,NaT,CA1,0,0_종일,10500,2_일반,5.083333,5.0,7,3
2,OA974876,XXXXX,C01,M,2015-05-01 00:00:00,NaT,CA1,0,0_종일,10500,2_일반,4.583333,5.0,6,3


In [36]:
# 고객정보에 규칙 이용 여부(routine_flg)까지 붙이기
customer_join4 = pd.merge(
    customer_join3,
    uselog_weekday[["customer_id", "routine_flg"]],
    on="customer_id",
    how="left"
)

# 가입일도 날짜 계산을 위해 날짜형으로 변경
customer_join4["start_date"] = pd.to_datetime(customer_join4["start_date"])

customer_join4.head(3)


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,campaign_name,mean,median,max,min,routine_flg
0,OA832399,XXXX,C01,F,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,4.833333,5.0,8,2,1
1,PL270116,XXXXX,C01,M,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,5.083333,5.0,7,3,1
2,OA974876,XXXXX,C01,M,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,4.583333,5.0,6,3,1


#### 회원이 몇 개월 동안 이용했는지 계산


In [37]:
# 두 날짜 사이의 년/월/일 차이를 계산하는 도구
from dateutil.relativedelta import relativedelta


In [38]:
# relativedelta 사용 예제
# 2000-11-13부터 2025-03-30까지 얼마나 지났는지 계산
a = relativedelta(
    pd.to_datetime("2025-03-30"),
    pd.to_datetime("2000-11-13")
)
a


relativedelta(years=+24, months=+4, days=+17)

In [39]:
# 계산 결과에서 "년" 부분만 확인
a.years


24

In [40]:
# 회원기간 계산에 사용할 마지막 날짜 만들기
# 탈퇴 회원 → 실제 탈퇴일 사용
# 현재 회원 → 탈퇴일이 없으므로 2019-04-30을 임시 기준일로 사용
customer_join4["calc_date"] = customer_join4["end_date"]
customer_join4["calc_date"] = customer_join4["calc_date"].fillna(
    pd.to_datetime("2019-04-30")
)

customer_join4.head(3)


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,campaign_name,mean,median,max,min,routine_flg,calc_date
0,OA832399,XXXX,C01,F,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,4.833333,5.0,8,2,1,2019-04-30
1,PL270116,XXXXX,C01,M,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,5.083333,5.0,7,3,1,2019-04-30
2,OA974876,XXXXX,C01,M,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,4.583333,5.0,6,3,1,2019-04-30


In [41]:
# 회원기간(개월 수)을 저장할 컬럼 만들기
customer_join4["membership_p"] = 0

# 고객 한 명씩 가입일부터 마지막 날짜까지의 개월 수 계산
for i in range(len(customer_join4)):
    delta = relativedelta(
        customer_join4.loc[i, "calc_date"],
        customer_join4.loc[i, "start_date"]
    )

    # 년도는 ×12 해서 개월로 바꾸고, 남은 개월을 더함
    customer_join4.loc[i, "membership_p"] = delta.years * 12 + delta.months

customer_join4.head(3)


,customer_id,name,class,gender,start_date,end_date,campaign_id,is_deleted,class_name,price,campaign_name,mean,median,max,min,routine_flg,calc_date,membership_p
0,OA832399,XXXX,C01,F,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,4.833333,5.0,8,2,1,2019-04-30,47
1,PL270116,XXXXX,C01,M,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,5.083333,5.0,7,3,1,2019-04-30,47
2,OA974876,XXXXX,C01,M,2015-05-01,NaT,CA1,0,0_종일,10500,2_일반,4.583333,5.0,6,3,1,2019-04-30,47


In [42]:
# 고객들의 월 이용횟수 통계를 한 번에 확인
customer_join4[["mean", "median", "max", "min"]].describe()


,mean,median,max,min
count,4192.000000,4192.000000,4192.000000,4192.000000
mean,5.333127,5.250596,7.823950,3.041269
std,1.777533,1.874874,2.168959,1.951565
min,1.000000,1.000000,1.000000,1.000000
25%,4.250000,4.000000,7.000000,2.000000
50%,5.000000,5.000000,8.000000,3.000000
75%,6.416667,6.500000,9.000000,4.000000
max,12.000000,12.000000,14.000000,12.000000


In [43]:
# 규칙 이용(0/1) 회원이 각각 몇 명인지 확인
customer_join4.groupby("routine_flg")[["customer_id"]].count()


,customer_id
routine_flg,
0,779
1,3413


In [44]:
# 탈퇴 회원만 추출해서 통계 확인
# is_deleted = 1 → 탈퇴 회원
customer_end = customer_join4.loc[
    customer_join4["is_deleted"] == 1
]

customer_end.describe()


,start_date,end_date,is_deleted,price,mean,median,max,min,routine_flg,calc_date,membership_p
count,1350,1350,1350.0,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000,1350.000000,1350,1350.000000
mean,2018-01-20 10:45:20,2018-10-17 16:23:28,1.0,8595.555556,3.865474,3.621852,6.461481,1.821481,0.456296,2018-10-17 16:23:28,8.026667
min,2016-05-01 00:00:00,2018-04-30 00:00:00,1.0,6000.000000,1.000000,1.000000,1.000000,1.000000,0.000000,2018-04-30 00:00:00,1.000000
25%,2017-10-01 00:00:00,2018-06-30 00:00:00,1.0,6000.000000,3.000000,3.000000,4.000000,1.000000,0.000000,2018-06-30 00:00:00,4.000000
50%,2018-03-01 00:00:00,2018-10-31 00:00:00,1.0,7500.000000,4.000000,4.000000,7.000000,2.000000,0.000000,2018-10-31 00:00:00,7.000000
75%,2018-06-03 00:00:00,2019-01-31 00:00:00,1.0,10500.000000,4.666667,4.500000,8.000000,2.000000,1.000000,2019-01-31 00:00:00,11.000000
max,2019-02-11 00:00:00,2019-03-31 00:00:00,1.0,10500.000000,9.000000,9.000000,13.000000,8.000000,1.000000,2019-03-31 00:00:00,23.000000
std,NaN,NaN,0.0,1949.163652,1.246385,1.270847,2.584021,0.976361,0.498271,NaN,5.033692


In [45]:
# 유지 회원만 추출해서 통계 확인
# is_deleted = 0 → 현재 회원
customer_stay = customer_join4.loc[
    customer_join4["is_deleted"] == 0
]

customer_stay.describe()


,start_date,end_date,is_deleted,price,mean,median,max,min,routine_flg,calc_date,membership_p
count,2842,0,2842.0,2842.000000,2842.000000,2842.000000,2842.000000,2842.000000,2842.000000,2842,2842.000000
mean,2017-04-03 22:42:28.627726848,NaT,0.0,8542.927516,6.030288,6.024279,8.471147,3.620690,0.984166,2019-04-30 00:00:00,23.970443
min,2015-05-01 00:00:00,NaT,0.0,6000.000000,3.166667,3.000000,5.000000,1.000000,0.000000,2019-04-30 00:00:00,1.000000
25%,2016-05-01 00:00:00,NaT,0.0,6000.000000,4.833333,5.000000,7.000000,2.000000,1.000000,2019-04-30 00:00:00,12.000000
50%,2017-04-01 00:00:00,NaT,0.0,7500.000000,5.583333,5.500000,8.000000,3.000000,1.000000,2019-04-30 00:00:00,24.000000
75%,2018-04-05 00:00:00,NaT,0.0,10500.000000,7.178030,7.000000,10.000000,5.000000,1.000000,2019-04-30 00:00:00,35.000000
max,2019-03-15 00:00:00,NaT,0.0,10500.000000,12.000000,12.000000,14.000000,12.000000,1.000000,2019-04-30 00:00:00,47.000000
std,NaN,NaN,0.0,1977.189779,1.553587,1.599765,1.571048,2.030488,0.124855,NaN,13.746761


In [46]:
# 결과 해석
# 탈퇴 회원은 유지 회원보다 월 평균/중앙/최대 이용횟수가 적은 편이다.
# routine_flg 평균도 유지 회원이 더 높다.
# 즉, 규칙적으로 자주 오는 회원일수록 계속 이용하는 경향을 확인할 수 있다.


In [47]:
# 최종 분석 데이터를 CSV 파일로 저장
customer_join4.to_csv(
    "data/customer_join4.csv",
    index=False,
    encoding="utf-8-sig"
)
